In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

# ==============================
# SIMULATION SETTINGS
# ==============================

n = 500
p = 1000
d = p + 1

m_true = 20
n_runs = 50

beta0 = 3.0
rho = 0.8

sigma_true = 3.0
sigma2_true = sigma_true**2

# This seed generates the fixed active set and beta.
beta_seed = 2026

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

# ==============================
# FIXED ACTIVE SET AND BETA
# ==============================

beta_rng = np.random.default_rng(beta_seed)

# Select 20 distinct active slope predictors.
true_active = beta_rng.choice(
    p,
    size=m_true,
    replace=False,
)
true_active.sort()

# Generate the nonzero coefficients once.
beta_slopes = np.zeros(p, dtype=np.float64)
beta_slopes[true_active] = beta_rng.normal(
    loc=0.0,
    scale=1.0,
    size=m_true,
)

# Intercept followed by 1,000 slope coefficients.
beta_full = np.concatenate(([beta0], beta_slopes))

# ==============================
# SAVE SHARED TRUE MODEL
# ==============================

is_true_active = np.zeros(p, dtype=np.int64)
is_true_active[true_active] = 1

true_model_df = pd.DataFrame({
    "design_column_0_based": np.arange(d),
    "variable_name": (
        ["Intercept"]
        + [f"X{j}" for j in range(1, p + 1)]
    ),
    "is_intercept": np.concatenate((
        [1],
        np.zeros(p, dtype=np.int64),
    )),
    "is_true_active": np.concatenate((
        [0],  # Intercept is separate from the 20 active predictors.
        is_true_active,
    )),
    "beta": beta_full,
})

true_model_file = output_dir / "fixed_beta_and_active_set.csv"

true_model_df.to_csv(
    true_model_file,
    index=False,
    float_format="%.17g",
)

print("Fixed active slope variables, using X1,...,X1000:")
print(true_active + 1)

print("\nFixed nonzero coefficients:")
print(
    true_model_df.loc[
        true_model_df["is_true_active"] == 1,
        ["variable_name", "beta"],
    ].to_string(index=False)
)

# ==============================
# GENERATE 50 DATASETS
# ==============================

scale = np.sqrt(1.0 - rho**2)

for seed in range(n_runs):
    rng = np.random.default_rng(seed)

    # --------------------------
    # Raw Gaussian AR(1) design
    # --------------------------

    X_raw = np.empty((n, p), dtype=np.float64)

    X_raw[:, 0] = rng.normal(
        loc=0.0,
        scale=1.0,
        size=n,
    )

    for j in range(1, p):
        X_raw[:, j] = (
            rho * X_raw[:, j - 1]
            + scale * rng.normal(
                loc=0.0,
                scale=1.0,
                size=n,
            )
        )

    # --------------------------
    # Standardise slopes only
    # --------------------------

    X_mean = X_raw.mean(axis=0)
    X_sd = X_raw.std(axis=0, ddof=1)

    if np.any(X_sd == 0.0):
        raise ValueError(
            f"Seed {seed} generated a constant predictor."
        )

    X = (
        X_raw - X_mean[None, :]
    ) / X_sd[None, :]

    # Add the untouched intercept column.
    X_full = np.column_stack((
        np.ones(n, dtype=np.float64),
        X,
    ))

    assert X_full.shape == (n, d)
    assert np.all(X_full[:, 0] == 1.0)

    # --------------------------
    # Generate noise and y
    # --------------------------

    noise = rng.normal(
        loc=0.0,
        scale=sigma_true,
        size=n,
    )

    y = X_full @ beta_full + noise

    # --------------------------
    # Save dataset CSV
    # --------------------------

    dataset_df = pd.DataFrame(
        X_full,
        columns=(
            ["Intercept"]
            + [f"X{j}" for j in range(1, p + 1)]
        ),
    )

    dataset_df["y"] = y
    dataset_df["noise"] = noise

    dataset_file = (
        output_dir / f"seed_{seed:02d}.csv"
    )

    dataset_df.to_csv(
        dataset_file,
        index=False,
        float_format="%.17g",
    )

    print(f"Saved seed {seed:02d}: {dataset_file.name}")

print("\nGeneration complete.")
print(f"Output folder: {output_dir.resolve()}")
print(f"Number of dataset CSV files: {n_runs}")
print(f"Shared beta file: {true_model_file.name}")